# CICIoT2023 — xử lý dữ liệu cho phân loại 8 lớp trên Kaggle

Chạy **Restart Session → Run All** sau khi gắn dataset. Notebook chỉ sử dụng `/kaggle/input` và ghi đầu ra vào `/kaggle/working/processed_data/<run_id>`.


## Phạm vi, nhận xét và các sửa đổi

Bài toán duy nhất gồm **Normal, BruteForce, DDoS, DoS, Mirai, Recon, Spoofing, Web-Based**, mã lần lượt **0–7**. Nhãn gốc chỉ dùng để ánh xạ và truy vết.

Bản trước đã làm đúng việc fit scaler trên train và không xóa hàng chỉ vì vượt ngưỡng IQR. Các điểm đã sửa:

- Kiểm tra schema, nhãn, NaN/Inf ở cả ba tập; median học từ train vẫn được dùng khi chỉ validation/test có giá trị thiếu.
- Khử trùng train theo **vector đặc trưng và nhãn 8 lớp**; kiểm tra giao nhau train/validation/test. Không suy luận rằng giữ nguyên split Kaggle sẽ bảo đảm đánh giá độc lập.
- Khi `CROSS_SPLIT_POLICY="purge_eval"`, giữ train, bỏ mẫu validation trùng train và mẫu test trùng train/validation được giữ. Quy tắc dựa trên đặc trưng, được áp dụng trước huấn luyện và ghi số dòng/support thay đổi. Đây là đánh giá trên các vector không trùng, không chứng minh độc lập theo thiết bị/phiên.
- Giữ và báo cáo các nhóm có cùng vector nhưng khác nhãn 8 lớp; đây có thể là giới hạn của đặc trưng, không tự kết luận nhãn sai.
- Cột hằng chỉ được xác định từ train. Tương quan dùng để khảo sát; không tuyên bố mô hình cây miễn nhiễm với tương quan hay một cờ giao thức là dấu hiệu duy nhất của một lớp.
- Log1p chỉ chọn trong danh sách đặc trưng có ý nghĩa phù hợp, dựa trên train; không log mã giao thức/cờ chỉ báo, không âm thầm cắt giá trị âm về 0. Xử lý được trường hợp không chọn cột log nào.
- Giới hạn cỡ mẫu vẽ; fit tiền xử lý trên train, xuất theo batch để hạn chế giữ nhiều ma trận lớn trong RAM.
- Lưu cả `tree_preprocessor.joblib` và `scaled_preprocessor.joblib`, mapping, schema, source-row ID, kiểm kê nguồn và kết quả audit. Báo cáo dùng số đo thực thay vì ghi cứng 46 features, 0 NaN hay hứa hẹn recall.

Output cũ đã được xóa để kết quả hiển thị sau Run All thuộc đúng mã mới. Chưa chạy lại notebook trên dữ liệu Kaggle.

Tham khảo kỹ thuật: [scikit-learn: tránh rò rỉ dữ liệu](https://scikit-learn.org/stable/common_pitfalls.html), [RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html), [NumPy log1p](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html). Lựa chọn log/scaler là cấu hình thực nghiệm cần đánh giá bằng validation, không mặc định làm mô hình tốt hơn.


## 1. Thư viện và cấu hình thực nghiệm


In [ ]:
import gc
import hashlib
import json
import platform
import time
import uuid
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler, StandardScaler

# Mọi tham số thay đổi tại đây trước khi Run All.
RANDOM_STATE = 42
READ_CHUNK_SIZE = 100_000
TRANSFORM_BATCH_SIZE = 25_000
AUDIT_SAMPLE_SIZE = 100_000
PLOT_SAMPLES_PER_CLASS = 2_000
CORR_THRESHOLD = 0.95
DROP_CONSTANT_FEATURES = True
DEDUPLICATE_TRAIN = True
CROSS_SPLIT_POLICY = "purge_eval"  # "purge_eval" hoặc "error"
REQUIRE_ALL_CLASSES_IN_EVAL = True
APPLY_LOG1P = True                 # Chỉ áp dụng cho nhánh scaled.
SKEW_THRESHOLD = 2.0
SCALING_METHOD = "RobustScaler"    # "RobustScaler" hoặc "StandardScaler"
SAVE_COMPAT_PICKLE = True          # Giữ X_*_scaled.pkl, y_*.pkl cho bước sau.
COMPUTE_SOURCE_SHA256 = True

CLASS_NAMES = [
    "Normal", "BruteForce", "DDoS", "DoS",
    "Mirai", "Recon", "Spoofing", "Web-Based",
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
RAW_FEATURE_NAMES = [
    "flow_duration", "Header_Length", "Protocol Type", "Duration", "Rate",
    "Srate", "Drate", "fin_flag_number", "syn_flag_number", "rst_flag_number",
    "psh_flag_number", "ack_flag_number", "ece_flag_number", "cwr_flag_number",
    "ack_count", "syn_count", "fin_count", "urg_count", "rst_count",
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP",
    "DHCP", "ARP", "ICMP", "IPv", "LLC", "Tot sum", "Min", "Max", "AVG",
    "Std", "Tot size", "IAT", "Number", "Magnitue", "Radius", "Covariance",
    "Variance", "Weight",
]
# Giữ nguyên chính tả "Magnitue" của CSV; không tự đổi tên thành Magnitude.
LABEL_GROUPS = {
    "Normal": ["BenignTraffic"],
    "BruteForce": ["DictionaryBruteForce"],
    "DDoS": [
        "DDoS-RSTFINFlood", "DDoS-PSHACK_Flood", "DDoS-SYN_Flood",
        "DDoS-UDP_Flood", "DDoS-TCP_Flood", "DDoS-ICMP_Flood",
        "DDoS-SynonymousIP_Flood", "DDoS-ACK_Fragmentation",
        "DDoS-UDP_Fragmentation", "DDoS-ICMP_Fragmentation",
        "DDoS-SlowLoris", "DDoS-HTTP_Flood",
    ],
    "DoS": ["DoS-UDP_Flood", "DoS-SYN_Flood", "DoS-TCP_Flood", "DoS-HTTP_Flood"],
    "Mirai": ["Mirai-greeth_flood", "Mirai-greip_flood", "Mirai-udpplain"],
    "Recon": ["Recon-HostDiscovery", "Recon-OSScan", "Recon-PortScan", "Recon-PingSweep", "VulnerabilityScan"],
    "Spoofing": ["MITM-ArpSpoofing", "DNS_Spoofing"],
    "Web-Based": ["BrowserHijacking", "CommandInjection", "SqlInjection", "XSS", "Backdoor_Malware", "Uploading_Attack"],
}
RAW_TO_CATEGORY = {
    raw: category for category, labels in LABEL_GROUPS.items() for raw in labels
}
# Chỉ xét log các đại lượng này; mã giao thức và cờ chỉ báo không được log.
LOG_CANDIDATE_FEATURES = [
    "flow_duration", "Header_Length", "Rate", "Srate", "Drate",
    "ack_count", "syn_count", "fin_count", "urg_count", "rst_count",
    "Tot sum", "Min", "Max", "AVG", "Std", "Tot size", "IAT",
    "Magnitue", "Radius", "Covariance",
]
if CROSS_SPLIT_POLICY not in {"purge_eval", "error"}:
    raise ValueError("CROSS_SPLIT_POLICY phải là purge_eval hoặc error.")
if SCALING_METHOD not in {"RobustScaler", "StandardScaler"}:
    raise ValueError("SCALING_METHOD không hợp lệ.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
plt.rcParams["figure.dpi"] = 120
SESSION_START = time.perf_counter()
print("Phiên bản:", {"Python": platform.python_version(), "pandas": pd.__version__,
                      "numpy": np.__version__, "sklearn": sklearn.__version__})
print("8 lớp và mã cố định:", CLASS_TO_ID)


In [ ]:
# Notebook này chủ đích chỉ chạy trên Kaggle.
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
if not KAGGLE_INPUT.is_dir() or not KAGGLE_WORKING.is_dir():
    raise RuntimeError("Hãy chạy notebook trên Kaggle và gắn dataset CICIoT2023.")

DATA_ROOT = Path("/kaggle/input/datasets/himadri07/ciciot2023/CICIOT23")

def split_paths(root):
    return {
        "train": root / "train/train.csv",
        "val": root / "validation/validation.csv",
        "test": root / "test/test.csv",
    }

if not all(path.is_file() for path in split_paths(DATA_ROOT).values()):
    candidates = []
    for path in KAGGLE_INPUT.rglob("train.csv"):
        root = path.parent.parent
        if all(candidate.is_file() for candidate in split_paths(root).values()):
            candidates.append(root)
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Không xác định được duy nhất bộ train/validation/test. "
            f"Hãy sửa DATA_ROOT. Các root tìm được: {candidates}"
        )
    DATA_ROOT = candidates[0]

SOURCE_PATHS = split_paths(DATA_ROOT)
RUN_ID = time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
OUTPUT_ROOT = KAGGLE_WORKING / "processed_data"
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=False)
(OUTPUT_DIR / "run_status.json").write_text(
    json.dumps({"run_id": RUN_ID, "status": "in_progress"}, indent=2), encoding="utf-8"
)
print("Nguồn:", {name: str(path) for name, path in SOURCE_PATHS.items()})
print("Đầu ra của lần chạy này:", OUTPUT_DIR)


In [ ]:
# Các hàm dùng chung: audit, fingerprint, giữ sample ID và xuất báo cáo.
def write_json(name, value):
    path = OUTPUT_DIR / name
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    return path


def source_checksum(path):
    if not COMPUTE_SOURCE_SHA256:
        return None
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def class_support(targets):
    counts = np.bincount(np.asarray(targets, dtype=np.int64), minlength=len(CLASS_NAMES))
    return {name: int(counts[idx]) for idx, name in enumerate(CLASS_NAMES)}


def support_table():
    records = []
    for split in SOURCE_PATHS:
        counts = class_support(metadata[split]["class_id"].to_numpy())
        total = len(metadata[split])
        for name in CLASS_NAMES:
            records.append({"split": split, "class_id": CLASS_TO_ID[name], "class_name": name,
                            "count": counts[name], "percentage": 100 * counts[name] / total if total else 0.0})
    return pd.DataFrame(records)


def ensure_class_coverage():
    for split in SOURCE_PATHS:
        if len(frames[split]) == 0:
            raise ValueError(f"{split} không còn dòng nào; xem các audit trong {OUTPUT_DIR}.")
        missing = [name for name, count in class_support(metadata[split]["class_id"]).items() if count == 0]
        if missing and (split == "train" or REQUIRE_ALL_CLASSES_IN_EVAL):
            raise ValueError(
                f"{split} thiếu các lớp {missing}. Cần xem lại dữ liệu/split theo nhóm. "
                "Không tự đổi mapping hoặc lấy mẫu test để bổ sung train."
            )
        if missing:
            print(f"LƯU Ý: {split} thiếu lớp {missing}; phải ghi support=0 khi đánh giá đủ 8 lớp.")


def retain_rows(split, mask):
    mask = np.asarray(mask, dtype=bool)
    if not mask.all():
        frames[split] = frames[split].loc[mask].reset_index(drop=True)
        metadata[split] = metadata[split].loc[mask].reset_index(drop=True)
    if len(frames[split]) != len(metadata[split]):
        raise RuntimeError("Dữ liệu và sample metadata lệch hàng.")


def fingerprints(frame, preprocessor=None, dtype="float64"):
    # BLAKE2b-128 trên vector số theo đúng thứ tự cột.
    # Chuẩn hóa -0.0 và NaN để cùng vector có cùng hash; xác suất va chạm hash không bằng 0.
    result = np.empty(len(frame), dtype="V16")
    for start in range(0, len(frame), TRANSFORM_BATCH_SIZE):
        stop = min(start + TRANSFORM_BATCH_SIZE, len(frame))
        part = frame.iloc[start:stop]
        values = preprocessor.transform(part) if preprocessor is not None else part.to_numpy()
        with np.errstate(over="ignore", invalid="ignore"):
            block = np.array(values, dtype=np.dtype(dtype).newbyteorder("<"), order="C", copy=True)
        if preprocessor is not None and not np.isfinite(block).all():
            raise ValueError("Biến đổi hoặc ép dtype tạo NaN/Inf; không xuất dữ liệu lỗi.")
        block[block == 0] = 0.0
        block[np.isnan(block)] = np.nan
        digests = [hashlib.blake2b(row.tobytes(), digest_size=16).digest() for row in block]
        result[start:stop] = np.asarray(digests, dtype="V16")
    return result


def audit_hashes(hashes, targets):
    unique_hashes = {split: np.unique(values) for split, values in hashes.items()}
    duplicates = {split: int(len(values) - len(unique_hashes[split])) for split, values in hashes.items()}
    overlaps = {}
    for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlaps[f"{left}__{right}"] = int(len(np.intersect1d(
            unique_hashes[left], unique_hashes[right], assume_unique=True
        )))
    all_hashes = np.concatenate([hashes[split] for split in SOURCE_PATHS])
    all_targets = np.concatenate([np.asarray(targets[split], dtype=np.uint8) for split in SOURCE_PATHS])
    order = np.argsort(all_hashes, kind="stable")
    ordered_hashes, ordered_targets = all_hashes[order], all_targets[order]
    different_label = (ordered_hashes[1:] == ordered_hashes[:-1]) & (ordered_targets[1:] != ordered_targets[:-1])
    conflict_keys = np.unique(ordered_hashes[1:][different_label])
    conflict_rows = {split: int(np.isin(values, conflict_keys).sum()) for split, values in hashes.items()}
    return {"duplicate_feature_rows": duplicates, "unique_overlap_groups": overlaps,
            "conflicting_8class_groups": int(len(conflict_keys)), "rows_in_conflicting_groups": conflict_rows,
            "conflict_policy": "keep_and_report", "fingerprint": "BLAKE2b-128; canonical numerical vector",
            "limitation": "Hash audit is not a proof of independence by device/session/time."}


def evaluation_masks(hash_views):
    # Ưu tiên train -> val -> test, hợp nhất mẫu trùng của các cách biểu diễn đầu vào.
    keep = {split: np.ones(len(frames[split]), dtype=bool) for split in SOURCE_PATHS}
    train_keys = {view: np.unique(values["train"]) for view, values in hash_views.items()}
    for view, values in hash_views.items():
        keep["val"] &= ~np.isin(values["val"], train_keys[view])
    for view, values in hash_views.items():
        retained_val_keys = np.unique(values["val"][keep["val"]])
        keep["test"] &= ~np.isin(values["test"], train_keys[view])
        keep["test"] &= ~np.isin(values["test"], retained_val_keys)
    removed = {}
    for split in SOURCE_PATHS:
        removed[split] = {
            "rows": int((~keep[split]).sum()),
            "by_class": class_support(metadata[split].loc[~keep[split], "class_id"].to_numpy()),
        }
    return keep, removed


def apply_overlap_policy(hash_views, report_name):
    targets = {split: metadata[split]["class_id"].to_numpy() for split in SOURCE_PATHS}
    report = {view: audit_hashes(values, targets) for view, values in hash_views.items()}
    keep, removed = evaluation_masks(hash_views)
    report["policy"] = CROSS_SPLIT_POLICY
    report["evaluation_rows_removed"] = removed
    write_json(report_name, report)
    print("Audit:", report_name, "| Số dòng cần bỏ:", {split: item["rows"] for split, item in removed.items()})
    if CROSS_SPLIT_POLICY == "error" and any(item["rows"] for item in removed.values()):
        raise ValueError(f"Có trùng xuyên tập; xem {report_name}. Chốt lại split hoặc chọn purge_eval trước Run All.")
    if CROSS_SPLIT_POLICY == "purge_eval":
        for split in SOURCE_PATHS:
            retain_rows(split, keep[split])
    return report


def train_audit_sample():
    sample = frames["train"].sample(n=min(AUDIT_SAMPLE_SIZE, len(frames["train"])), random_state=RANDOM_STATE)
    return sample.loc[:, final_feature_names].fillna(train_medians)


## 2. Đọc CSV, xác thực schema và ánh xạ 8 lớp

Đọc theo chunk và dùng categorical cho nhãn để giảm bản sao chuỗi. Đặc trưng giữ `float64` ở giai đoạn kiểm tra để tránh khử trùng nhầm vì làm tròn `float32`. Mọi cột nhãn nằm trong metadata riêng, không nằm trong ma trận feature. `source_row_index` là chỉ số dòng dữ liệu 0-based trong CSV nguồn; sample ID đầy đủ là `<split>:<source_row_index>`.


In [ ]:
frames, metadata, source_manifest = {}, {}, {}
load_start = time.perf_counter()
for split, path in SOURCE_PATHS.items():
    started = time.perf_counter()
    header = pd.read_csv(path, nrows=0).columns.tolist()
    expected = set(RAW_FEATURE_NAMES + ["label"])
    if len(header) != len(expected) or set(header) != expected:
        raise ValueError(f"Schema {split} không khớp: thiếu={sorted(expected-set(header))}, thừa={sorted(set(header)-expected)}")
    chunks = []
    dtype = {name: "float64" for name in RAW_FEATURE_NAMES}
    dtype["label"] = "string"
    for chunk in pd.read_csv(path, dtype=dtype, chunksize=READ_CHUNK_SIZE):
        chunks.append(chunk.loc[:, RAW_FEATURE_NAMES + ["label"]])
    if not chunks:
        raise ValueError(f"CSV {split} không có dữ liệu.")
    data = pd.concat(chunks, ignore_index=True)
    del chunks
    labels = data.pop("label").str.strip()
    if labels.isna().any() or labels.eq("").any():
        raise ValueError(f"{split} có nhãn thiếu/rỗng; không điền median vào nhãn.")
    categories = labels.map(RAW_TO_CATEGORY)
    if categories.isna().any():
        raise ValueError(f"{split} có nhãn chưa ánh xạ: {labels[categories.isna()].value_counts().to_dict()}")
    encoded = categories.map(CLASS_TO_ID).to_numpy(dtype=np.uint8)
    frames[split] = data
    metadata[split] = pd.DataFrame({
        "source_row_index": np.arange(len(data), dtype=np.int64),
        "original_label": pd.Categorical(labels),
        "class_id": encoded,
    })
    source_manifest[split] = {
        "path": str(path), "bytes": int(path.stat().st_size), "sha256": source_checksum(path),
        "rows": int(len(data)), "num_raw_features": len(RAW_FEATURE_NAMES),
        "raw_labels_present": sorted(labels.unique().tolist()), "class_support": class_support(encoded),
        "load_and_checksum_seconds": round(time.perf_counter() - started, 3),
        "dataframe_ram_mb": round(float(data.memory_usage(deep=True).sum() / 1024**2), 3),
    }
    print(split, data.shape, "| RAM MiB:", source_manifest[split]["dataframe_ram_mb"])
    del data, labels, categories, encoded
    gc.collect()
load_time = time.perf_counter() - load_start
write_json("source_manifest.json", source_manifest)
support_table().to_csv(OUTPUT_DIR / "class_support_source.csv", index=False)
ensure_class_coverage()
print(f"Đọc và kiểm kê nguồn: {load_time:.2f} giây")


## 3. Kiểm tra NaN/Inf

Đổi Inf thành NaN ở cả ba tập; ghi số lượng theo cột. Median sẽ được tính từ train sau khử trùng, và áp dụng ở các transformer cho mọi tập, kể cả khi chỉ validation/test có NaN. Cột hoàn toàn thiếu ở train sẽ làm notebook dừng để kiểm tra nguồn, thay vì điền một giá trị tùy ý.


In [ ]:
quality_records = []
for split, frame in frames.items():
    for feature in RAW_FEATURE_NAMES:
        values = frame[feature].to_numpy()
        inf_mask = np.isinf(values)
        n_inf, n_nan = int(inf_mask.sum()), int(np.isnan(values).sum())
        quality_records.append({"split": split, "feature": feature,
                                "nan_before": n_nan, "inf_before": n_inf,
                                "missing_after_inf_to_nan": n_nan + n_inf})
        if n_inf:
            frame.loc[inf_mask, feature] = np.nan
quality_audit = pd.DataFrame(quality_records)
quality_audit.to_csv(OUTPUT_DIR / "missing_inf_audit.csv", index=False)
display(quality_audit.groupby("split")[["nan_before", "inf_before", "missing_after_inf_to_nan"]].sum())
print("NaN sẽ được điền bằng thống kê train trong transformer; nhãn đã được kiểm tra riêng.")


## 4. Khử trùng train và kiểm tra giao nhau các tập

Fingerprint được tính trên 46 đặc trưng, sau khi đổi Inf thành NaN và trước điền median/log/scale. Khử trùng train theo `(fingerprint, class_id)` để giữ các trường hợp cùng vector nhưng khác nhãn 8 lớp. Báo cáo nhóm nhãn mâu thuẫn; không tự kết luận tất cả là nhãn sai.

Mặc định **giữ train → bỏ mẫu validation trùng train → bỏ mẫu test trùng train/validation còn lại**. Giữ các dòng lặp bên trong từng tập validation/test. Cách này thay đổi phân bố đánh giá; lưu support trước/sau và công bố rõ trong báo cáo. Khi không chấp nhận thay đổi split, chọn `CROSS_SPLIT_POLICY="error"` để dừng và thiết kế lại split theo nhóm.


In [ ]:
raw_hashes = {split: fingerprints(frame) for split, frame in frames.items()}
raw_targets = {split: metadata[split]["class_id"].to_numpy() for split in SOURCE_PATHS}
raw_split_audit_before = audit_hashes(raw_hashes, raw_targets)
write_json("raw_split_audit_before.json", raw_split_audit_before)
print("Giao nhau ban đầu:", raw_split_audit_before["unique_overlap_groups"])
print("Nhóm vector có nhãn 8 lớp khác nhau:", raw_split_audit_before["conflicting_8class_groups"])

train_dedup_report = {"enabled": DEDUPLICATE_TRAIN, "rows_removed": 0,
                      "key": "46-feature fingerprint + 8-class target"}
if DEDUPLICATE_TRAIN:
    keys = np.empty(len(frames["train"]), dtype=[("features", "V16"), ("class_id", "u1")])
    keys["features"] = raw_hashes["train"]
    keys["class_id"] = metadata["train"]["class_id"].to_numpy()
    unique_keys, first_indices = np.unique(keys, return_index=True)
    keep_train = np.zeros(len(keys), dtype=bool)
    keep_train[first_indices] = True
    train_dedup_report["rows_removed"] = int((~keep_train).sum())
    train_dedup_report["removed_by_class"] = class_support(metadata["train"].loc[~keep_train, "class_id"])
    retain_rows("train", keep_train)
    raw_hashes["train"] = raw_hashes["train"][keep_train]
    del keys, unique_keys, first_indices, keep_train
write_json("train_deduplication.json", train_dedup_report)
raw_overlap_report = apply_overlap_policy({"original_numeric_features": raw_hashes}, "raw_split_overlap_policy.json")
support_table().to_csv(OUTPUT_DIR / "class_support_after_raw_audit.csv", index=False)
ensure_class_coverage()
del raw_hashes, raw_targets
gc.collect()
print("Số dòng khử trùng train:", train_dedup_report["rows_removed"])


## 5. Median từ train và kiểm tra cột hằng

Median được tính trên train đã khử trùng. Cột hằng được xác định bằng số giá trị phân biệt trên train sau điền thiếu, không dùng validation/test để chọn feature. Không dùng tỷ lệ 99% để xóa cột: một cột hiếm vẫn có thể chứa thông tin.

Nếu `DROP_CONSTANT_FEATURES=True`, cột hằng sẽ bị bỏ ở mọi nhánh theo cùng schema. Số feature cuối phụ thuộc dữ liệu thực chạy; không ghi cố định 46 khi đã bỏ cột. Cột hằng trên train nhưng thay đổi ở dữ liệu khác là một giới hạn cần nêu.


In [ ]:
train_medians = {}
constant_records = []
for feature in RAW_FEATURE_NAMES:
    column = frames["train"][feature]
    if column.notna().sum() == 0:
        raise ValueError(f"{feature} hoàn toàn thiếu trong train; kiểm tra nguồn dữ liệu.")
    median = float(column.median())
    if not np.isfinite(median):
        raise ValueError(f"Median train của {feature} không hữu hạn.")
    train_medians[feature] = median
    # Median lấy từ các giá trị đã có, nên điền median không tạo giá trị phân biệt mới.
    n_unique = int(column.nunique(dropna=True))
    constant_records.append({"feature": feature, "unique_nonmissing_train": n_unique,
                             "median_train": median, "constant_on_train": n_unique == 1})
constant_audit = pd.DataFrame(constant_records)
cols_to_drop_const = constant_audit.loc[constant_audit["constant_on_train"], "feature"].tolist() if DROP_CONSTANT_FEATURES else []
final_feature_names = [feature for feature in RAW_FEATURE_NAMES if feature not in cols_to_drop_const]
if not final_feature_names:
    raise ValueError("Không còn feature sau kiểm tra cột hằng.")
constant_audit.to_csv(OUTPUT_DIR / "constant_feature_audit.csv", index=False)
write_json("train_medians.json", train_medians)
print("Cột hằng bỏ theo train:", cols_to_drop_const)
print("Đặc trưng đầu vào:", len(RAW_FEATURE_NAMES), "| Đặc trưng đầu ra:", len(final_feature_names))


## 6. Khảo sát tương quan trên mẫu train

Mẫu tối đa `AUDIT_SAMPLE_SIZE` dòng, seed cố định; đây là ước lượng tương quan trên mẫu, không phải toàn bộ train. Giữ các feature chưa bị loại ở bước cột hằng; chưa tự bỏ cột theo tương quan. Việc giảm feature cần được đối sánh bằng validation; tương quan cũng ảnh hưởng cách diễn giải SHAP.


In [ ]:
audit_sample = train_audit_sample()
corr_matrix = audit_sample.corr()
correlated_pairs = []
for i, left in enumerate(final_feature_names):
    for right in final_feature_names[i + 1:]:
        value = float(corr_matrix.loc[left, right])
        if np.isfinite(value) and abs(value) >= CORR_THRESHOLD:
            correlated_pairs.append({"feature_1": left, "feature_2": right,
                                     "correlation": value, "abs_correlation": abs(value)})
df_corr_pairs = pd.DataFrame(correlated_pairs, columns=["feature_1", "feature_2", "correlation", "abs_correlation"])
if not df_corr_pairs.empty:
    df_corr_pairs = df_corr_pairs.sort_values("abs_correlation", ascending=False)
df_corr_pairs.to_csv(OUTPUT_DIR / "correlated_feature_pairs.csv", index=False)
corr_matrix.to_csv(OUTPUT_DIR / "correlation_matrix_train_sample.csv")
cols_to_drop_corr = []
print("Cỡ mẫu tương quan:", len(audit_sample), "| Cặp |r| >=", CORR_THRESHOLD, ":", len(df_corr_pairs))
display(df_corr_pairs.head(12))
fig, ax = plt.subplots(figsize=(15, 12))
mask = np.triu(np.ones(corr_matrix.shape, dtype=bool))
image = ax.imshow(np.ma.array(corr_matrix.to_numpy(), mask=mask), vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(final_feature_names)), labels=final_feature_names, rotation=90, fontsize=7)
ax.set_yticks(range(len(final_feature_names)), labels=final_feature_names, fontsize=7)
ax.set_title(f"Train feature correlation — sample n={len(audit_sample):,}, seed={RANDOM_STATE}")
fig.colorbar(image, ax=ax, shrink=0.7)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "correlation_train_sample.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)


## 7. Phân bố 8 lớp và mapping cố định

Mapping được khai báo rõ từ đầu: **Normal=0, BruteForce=1, DDoS=2, DoS=3, Mirai=4, Recon=5, Spoofing=6, Web-Based=7**. Không dùng `LabelEncoder` để suy ra thứ tự: công cụ đó sắp xếp tên lớp.

Các bảng support luôn gồm đủ 8 lớp. Biểu đồ train dùng tỷ lệ có ít nhất hai chữ số thập phân; tỷ số mất cân bằng tính trên 8 nhóm. Nhãn gốc chỉ lưu trong sample metadata.


In [ ]:
class_distribution = support_table()
class_distribution.to_csv(OUTPUT_DIR / "class_support_before_effective_audit.csv", index=False)
display(class_distribution)
write_json("label_mapping.json", {"class_names": CLASS_NAMES, "class_to_id": CLASS_TO_ID,
                                  "id_to_class": {str(idx): name for idx, name in enumerate(CLASS_NAMES)},
                                  "original_label_to_class": RAW_TO_CATEGORY})
train_distribution = class_distribution.loc[class_distribution["split"] == "train"]
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(train_distribution["class_name"], train_distribution["count"], color=plt.cm.tab10.colors[:8])
for bar, row in zip(bars, train_distribution.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{row.count:,}\n({row.percentage:.2f}%)", ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, max(train_distribution["count"]) * 1.18)
ax.set_title("Train — phân bố 8 lớp sau khử trùng")
ax.set_ylabel("Số dòng dữ liệu")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "class_distribution_8_classes_train.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
counts = train_distribution["count"].to_numpy()
print(f"Tỷ số lớp lớn nhất/nhỏ nhất (8 lớp): {counts.max()/counts.min():,.2f}:1")


## 8. Khảo sát đuôi dài và ngưỡng IQR

IQR/skewness được ước lượng trên mẫu train đã điền bằng median train. Khi IQR bằng 0, nhiều giá trị khác 0 có thể nằm ngoài khoảng IQR; điều đó không tự chứng minh dữ liệu sai hay là tấn công. Notebook không xóa/cắt hàng theo ngưỡng IQR.

Boxplot lấy tối đa `PLOT_SAMPLES_PER_CLASS` mỗi lớp, giữ lớp hiếm có mẫu. Mẫu này phục vụ so sánh hình dạng theo lớp, không đại diện tỷ lệ lớp tự nhiên.


In [ ]:
outlier_records = []
for feature in final_feature_names:
    column = audit_sample[feature]
    q25, q75 = column.quantile([0.25, 0.75]).to_numpy()
    iqr = float(q75 - q25)
    outside = (column < q25 - 1.5 * iqr) | (column > q75 + 1.5 * iqr)
    outlier_records.append({
        "feature": feature, "skewness_sample": float(column.skew()), "iqr_sample": iqr,
        "outside_iqr_count_sample": int(outside.sum()), "outside_iqr_pct_sample": float(outside.mean() * 100),
        "sample_size": len(column), "unique_nonmissing_train": int(frames["train"][feature].nunique(dropna=True)),
        "min_train": float(frames["train"][feature].min()), "max_train": float(frames["train"][feature].max()),
    })
df_outlier_summary = pd.DataFrame(outlier_records)
df_outlier_summary.to_csv(OUTPUT_DIR / "outlier_audit_train_sample.csv", index=False)
display(df_outlier_summary.sort_values("outside_iqr_pct_sample", ascending=False).head(12))

plot_features = [feature for feature in ["Rate", "Tot sum", "IAT", "flow_duration"] if feature in final_feature_names]
plot_indices = []
y_train_current = metadata["train"]["class_id"].to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
for class_id in range(len(CLASS_NAMES)):
    indices = np.flatnonzero(y_train_current == class_id)
    plot_indices.append(rng.choice(indices, size=min(len(indices), PLOT_SAMPLES_PER_CLASS), replace=False))
plot_indices = np.concatenate(plot_indices)
plot_sample = frames["train"].iloc[plot_indices].loc[:, plot_features].fillna(train_medians)
plot_targets = y_train_current[plot_indices]
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, feature in zip(axes.flat, plot_features):
    arrays = [plot_sample.loc[plot_targets == class_id, feature].to_numpy() for class_id in range(8)]
    ax.boxplot(arrays, showfliers=True, flierprops={"markersize": 2})
    ax.set_xticks(range(1, 9), labels=CLASS_NAMES, rotation=25, fontsize=8)
    ax.set_yscale("symlog")
    ax.set_title(feature)
    ax.set_ylabel("Giá trị gốc; trục symlog")
for ax in list(axes.flat)[len(plot_features):]:
    ax.set_visible(False)
fig.suptitle(f"Train boxplots — tối đa {PLOT_SAMPLES_PER_CLASS:,} mẫu/lớp; seed={RANDOM_STATE}")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "boxplots_8_classes_train_sample.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
del plot_sample, plot_indices, plot_targets, plot_features, y_train_current
# Không giữ thêm bản sao train toàn bộ chỉ để vẽ.
gc.collect()


## 9. Chọn log1p cho nhánh scaled

Chỉ chọn cột trong danh sách cho phép, có trên hai giá trị phân biệt, skewness trên mẫu train lớn hơn ngưỡng và không có giá trị âm ở train. Các đặc trưng có giá trị âm được giữ ở thang gốc. Nếu validation/test có giá trị âm trong cột đã chọn log, dừng và báo lỗi để kiểm tra; không thay chính sách theo điểm test.

Danh sách cột log được đóng băng và lưu trong cấu hình. Nhánh dữ liệu sạch cho cây giữ giá trị gốc sau imputation; nhánh scaled áp dụng log có chọn lọc rồi scaler. Cả hai giữ cùng thứ tự feature và cùng nhãn 8 lớp.


In [ ]:
cols_to_log1p, log_selection_records = [], []
for row in df_outlier_summary.itertuples():
    selected = bool(
        APPLY_LOG1P and row.feature in LOG_CANDIDATE_FEATURES
        and np.isfinite(row.skewness_sample) and abs(row.skewness_sample) > SKEW_THRESHOLD
        and row.unique_nonmissing_train > 2 and row.min_train >= 0
    )
    if selected:
        cols_to_log1p.append(row.feature)
    log_selection_records.append({"feature": row.feature, "selected": selected,
                                  "allowed_by_semantics": row.feature in LOG_CANDIDATE_FEATURES,
                                  "skewness_sample": float(row.skewness_sample), "min_train": float(row.min_train)})
pd.DataFrame(log_selection_records).to_csv(OUTPUT_DIR / "log1p_selection.csv", index=False)
for split, frame in frames.items():
    negatives = {feature: int((frame[feature] < 0).sum()) for feature in cols_to_log1p if (frame[feature] < 0).any()}
    if negatives:
        raise ValueError(f"{split} có giá trị âm ở cột log: {negatives}. Kiểm tra dữ liệu; không cắt ngầm về 0.")
print("Các cột log1p của nhánh scaled:", cols_to_log1p)
if cols_to_log1p:
    demo_feature = "Rate" if "Rate" in cols_to_log1p else cols_to_log1p[0]
    before = audit_sample[demo_feature].to_numpy()
    after = np.log1p(before)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(before, bins=50)
    axes[1].hist(after, bins=50)
    for ax in axes:
        ax.set_yscale("log")
        ax.set_ylabel("Số mẫu; trục log")
    axes[0].set_title(f"{demo_feature} — gốc")
    axes[1].set_title(f"{demo_feature} — log1p")
    fig.suptitle(f"Mẫu train n={len(before):,}; seed={RANDOM_STATE}")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "log1p_example_train_sample.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    del before, after
else:
    print("Không có cột phù hợp log1p; nhánh scaled vẫn điền thiếu và scale bình thường.")
del audit_sample
gc.collect()


## 10. Fit và đóng gói tiền xử lý dùng lại

Mỗi cột được điền bằng median train đã lưu. Nhánh `tree_preprocessor` chọn feature và điền thiếu, giữ đơn vị gốc. Nhánh `scaled_preprocessor` thêm log cho cột đã chọn và scaler cho đặc trưng có trên hai giá trị phân biệt; cờ hai giá trị giữ thang gốc. Các scaler được fit trên **toàn bộ train hiện tại**, không dùng validation/test.

`RobustScaler` dùng median/IQR; IQR bằng 0 không bảo đảm kháng mọi ngoại lai. Bảng tham số ghi các cột có IQR bằng 0 để giải thích giới hạn. Cấu hình scaler cần được đối sánh bằng validation. Không dùng median toàn ma trận như bằng chứng không có rò rỉ.

Các transformer dùng lớp/hàm có sẵn từ scikit-learn/NumPy để file joblib nạp được trong notebook khác mà không cần định nghĩa lại hàm tùy biến.


In [ ]:
tree_transformers, scaled_transformers = [], []
scaled_feature_names = []
for i, feature in enumerate(final_feature_names):
    name = f"feature_{i:02d}"
    def median_imputer():
        return SimpleImputer(strategy="constant", fill_value=train_medians[feature], keep_empty_features=True)
    tree_transformers.append((name, median_imputer(), [feature]))
    steps = [("imputer", median_imputer())]
    if feature in cols_to_log1p:
        steps.append(("log1p", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")))
    unique_count = int(constant_audit.loc[constant_audit["feature"] == feature, "unique_nonmissing_train"].iloc[0])
    if unique_count > 2:
        scaling = RobustScaler() if SCALING_METHOD == "RobustScaler" else StandardScaler()
        steps.append(("scaler", scaling))
        scaled_feature_names.append(feature)
    scaled_transformers.append((name, Pipeline(steps), [feature]))

tree_preprocessor = ColumnTransformer(tree_transformers, remainder="drop", n_jobs=1, verbose_feature_names_out=False)
scaled_preprocessor = ColumnTransformer(scaled_transformers, remainder="drop", n_jobs=1, verbose_feature_names_out=False)
fit_start = time.perf_counter()
# Nhánh tree chỉ có imputer hằng với median đã tính từ toàn train; 1 hàng dùng để nhận schema.
tree_preprocessor.fit(frames["train"].iloc[:1])
# Nhánh scaled học thống kê scaler từ toàn train. n_jobs=1 giảm bản sao dữ liệu song song.
scaled_preprocessor.fit(frames["train"])
preprocessor_fit_seconds = time.perf_counter() - fit_start
scaler_records = []
for i, feature in enumerate(final_feature_names):
    pipeline = scaled_preprocessor.named_transformers_[f"feature_{i:02d}"]
    record = {"feature": feature, "log1p": feature in cols_to_log1p,
              "scaled": feature in scaled_feature_names, "median_imputation": train_medians[feature]}
    if "scaler" in pipeline.named_steps:
        fitted = pipeline.named_steps["scaler"]
        record["scale"] = float(fitted.scale_[0])
        center = fitted.center_[0] if SCALING_METHOD == "RobustScaler" else fitted.mean_[0]
        record["center_or_mean"] = float(center)
        # Kiểm tra đúng IQR trên toàn train ở thang đầu vào scaler, từng cột để giảm RAM.
        train_column = frames["train"][feature].fillna(train_medians[feature])
        if feature in cols_to_log1p:
            train_column = np.log1p(train_column)
        q25, q75 = train_column.quantile([0.25, 0.75]).to_numpy()
        record["iqr_train_before_scaling"] = float(q75 - q25)
        record["zero_iqr_train"] = bool(q75 == q25)
        del train_column
    scaler_records.append(record)
pd.DataFrame(scaler_records).to_csv(OUTPUT_DIR / "scaler_parameters.csv", index=False)
joblib.dump(tree_preprocessor, OUTPUT_DIR / "tree_preprocessor.joblib")
joblib.dump(scaled_preprocessor, OUTPUT_DIR / "scaled_preprocessor.joblib")
print(f"Fit scaler trên train: {preprocessor_fit_seconds:.2f} giây")
print("Thứ tự output chung:", final_feature_names)
print("Dùng transformer.transform(feature_dataframe) cho dữ liệu mới; không fit lại.")
gc.collect()


## 11. Audit đầu vào mô hình và lưu artifact theo batch

Sau bỏ cột hằng/imputation, những vector khác nhau ban đầu có thể trở thành giống nhau. Ép scaled sang float32 cũng có thể tạo vector trùng. Kiểm tra lại giao nhau trên **nhánh tree float64** và **nhánh scaled float32** thực xuất; áp dụng cùng mask loại mẫu evaluation cho cả hai nhánh để mọi model dùng cùng split. Train không đổi ở bước này nên thống kê tiền xử lý vẫn thuộc train đã chốt.

Xuất `.npy` bằng memmap theo batch. Nhánh tree có `X_*_raw.npy` (đã chọn cột/điền thiếu, thang gốc), nhánh scaled có `X_*_scaled.npy`. Nếu bật tương thích, lưu thêm `X_*_scaled.pkl`; nhãn luôn có `y_*.pkl` và `y_*.npy`. Metadata và mọi ma trận giữ cùng thứ tự dòng. Các file có thể lớn; tắt `SAVE_COMPAT_PICKLE` để giảm dung lượng xuất trùng.

`run_status.json` chỉ ghi `complete` ở cell tổng kết cuối. Nếu một cell lỗi, thư mục lần chạy đó chưa được nghiệm thu; sửa cấu hình rồi Restart/Run All để tạo run mới.


In [ ]:
print("Đang audit vector sau preprocessing; bước này có thể mất thời gian trên hàng triệu dòng.")
effective_hash_views = {
    "tree_float64": {split: fingerprints(frame, tree_preprocessor, "float64") for split, frame in frames.items()},
    "scaled_float32": {split: fingerprints(frame, scaled_preprocessor, "float32") for split, frame in frames.items()},
}
effective_overlap_report = apply_overlap_policy(effective_hash_views, "effective_input_overlap_policy.json")
final_support = support_table()
final_support.to_csv(OUTPUT_DIR / "class_support_final.csv", index=False)
ensure_class_coverage()
del effective_hash_views
gc.collect()

artifact_manifest = {}
export_start = time.perf_counter()
for split in SOURCE_PATHS:
    frame = frames[split]
    target = metadata[split]["class_id"].to_numpy(dtype=np.uint8)
    shape = (len(frame), len(final_feature_names))
    raw_path = OUTPUT_DIR / f"X_{split}_raw.npy"
    scaled_path = OUTPUT_DIR / f"X_{split}_scaled.npy"
    raw_matrix = np.lib.format.open_memmap(raw_path, mode="w+", dtype=np.float64, shape=shape)
    scaled_matrix = np.lib.format.open_memmap(scaled_path, mode="w+", dtype=np.float32, shape=shape)
    for start in range(0, len(frame), TRANSFORM_BATCH_SIZE):
        stop = min(start + TRANSFORM_BATCH_SIZE, len(frame))
        part = frame.iloc[start:stop]
        raw_values = tree_preprocessor.transform(part)
        with np.errstate(over="ignore", invalid="ignore"):
            scaled_values = scaled_preprocessor.transform(part).astype(np.float32)
        if not np.isfinite(raw_values).all() or not np.isfinite(scaled_values).all():
            raise ValueError(f"{split} batch {start}:{stop} có NaN/Inf sau biến đổi.")
        raw_matrix[start:stop] = raw_values
        scaled_matrix[start:stop] = scaled_values
    raw_matrix.flush()
    scaled_matrix.flush()
    del raw_matrix, scaled_matrix, raw_values, scaled_values, part
    np.save(OUTPUT_DIR / f"y_{split}.npy", target, allow_pickle=False)
    pd.Series(target, name="class_id").to_pickle(OUTPUT_DIR / f"y_{split}.pkl")
    metadata[split].to_pickle(OUTPUT_DIR / f"sample_metadata_{split}.pkl")
    if SAVE_COMPAT_PICKLE:
        mapped = np.load(scaled_path, mmap_mode="r", allow_pickle=False)
        pd.DataFrame(mapped, columns=final_feature_names, copy=False).to_pickle(OUTPUT_DIR / f"X_{split}_scaled.pkl")
        del mapped
    artifact_manifest[split] = {
        "shape": list(shape), "class_support": class_support(target),
        "raw_file": raw_path.name, "raw_dtype": "float64",
        "scaled_file": scaled_path.name, "scaled_dtype": "float32",
        "target_file": f"y_{split}.npy", "metadata_file": f"sample_metadata_{split}.pkl",
        "compat_scaled_pickle": f"X_{split}_scaled.pkl" if SAVE_COMPAT_PICKLE else None,
    }
    print(split, shape, "| Export xong; y và source row ID cùng thứ tự.")
    del frames[split]
    del frame, target
    gc.collect()
export_seconds = time.perf_counter() - export_start

(OUTPUT_DIR / "feature_names.txt").write_text("\n".join(final_feature_names) + "\n", encoding="utf-8")
write_json("feature_schema.json", {
    "input_features": RAW_FEATURE_NAMES, "input_dtype": "float64", "label_input_to_estimator": False,
    "output_features": final_feature_names, "dropped_constant_features": cols_to_drop_const,
    "class_names": CLASS_NAMES, "class_to_id": CLASS_TO_ID,
    "sample_id_format": "<source_split>:<source_row_index>; 0-based data row index",
})
preprocessing_config = {
    "project": "CICIoT2023 supervised 8-class classification", "run_id": RUN_ID,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "seed": RANDOM_STATE,
    "source_root": str(DATA_ROOT), "source_manifest": "source_manifest.json",
    "class_names": CLASS_NAMES, "class_mapping": CLASS_TO_ID,
    "num_original_features": len(RAW_FEATURE_NAMES), "num_final_features": len(final_feature_names),
    "final_features": final_feature_names, "constant_features_dropped": cols_to_drop_const,
    "correlated_features_dropped": cols_to_drop_corr,
    "imputation": {"method": "train median", "medians": train_medians, "fitted_on": "deduplicated train"},
    "outlier_policy": "audit only; no row removal or clipping by IQR",
    "log1p": {"enabled_scaled_branch": APPLY_LOG1P, "features": cols_to_log1p,
               "threshold": SKEW_THRESHOLD, "selection_sample_size": min(AUDIT_SAMPLE_SIZE, source_manifest["train"]["rows"] - train_dedup_report["rows_removed"]),
               "negative_policy": "skip train-negative candidates; error for negative input in selected columns"},
    "scaling": {"method": SCALING_METHOD, "features": scaled_feature_names, "fit_split": "train",
                "two_value_features": "unscaled", "parameters_file": "scaler_parameters.csv"},
    "preprocessors": {"tree": "tree_preprocessor.joblib", "scaled": "scaled_preprocessor.joblib"},
    "split_policy": {"mode": CROSS_SPLIT_POLICY, "priority": ["train", "val", "test"],
                     "raw_report": "raw_split_overlap_policy.json", "effective_report": "effective_input_overlap_policy.json",
                     "within_eval_duplicates": "retained", "conflicting_labels": "kept_and_reported",
                     "limitation": "No validated device/session/time independence; evaluation distribution changed by purging."},
    "train_deduplication": train_dedup_report, "artifacts": artifact_manifest,
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "sklearn": sklearn.__version__, "joblib": joblib.__version__},
    "timing_seconds": {"load_and_source_audit": load_time, "preprocessor_fit": preprocessor_fit_seconds, "export": export_seconds},
}
write_json("preprocessing_config.json", preprocessing_config)
print("Artifacts:", OUTPUT_DIR)


In [ ]:
# Chỉ cell này xác nhận bộ artifact của lần chạy đã hoàn tất.
display(final_support)
print("Đặc trưng gốc:", len(RAW_FEATURE_NAMES))
print("Đặc trưng đầu ra:", len(final_feature_names), "| Cột hằng bỏ:", cols_to_drop_const)
print("Dòng trùng train đã bỏ:", train_dedup_report["rows_removed"])
print("Dòng evaluation bỏ ở audit đầu:", raw_overlap_report["evaluation_rows_removed"])
print("Dòng evaluation bỏ thêm sau preprocessing:", effective_overlap_report["evaluation_rows_removed"])
print("Số NaN/Inf nguồn theo split:")
display(quality_audit.groupby("split")[["nan_before", "inf_before"]].sum())
print("Cột log1p (scaled):", cols_to_log1p)
print("Scaler:", SCALING_METHOD, "| Fit trên train; cờ hai giá trị giữ thang gốc.")
print("Tổng thời gian giây:", round(time.perf_counter() - SESSION_START, 2))
print("Tổng dung lượng file MiB:", round(sum(path.stat().st_size for path in OUTPUT_DIR.rglob("*") if path.is_file()) / 1024**2, 2))
write_json("run_status.json", {"run_id": RUN_ID, "status": "complete",
                              "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                              "total_seconds": round(time.perf_counter() - SESSION_START, 3)})
(OUTPUT_ROOT / "latest_run.json").write_text(
    json.dumps({"run_id": RUN_ID, "path": str(OUTPUT_DIR), "status": "complete"}, indent=2), encoding="utf-8"
)
print("BỘ TIỀN XỬ LÝ ĐÃ XUẤT XONG:", OUTPUT_DIR)
print("Bước tiếp theo: đối sánh ít nhất 3 mô hình phân loại 8 lớp trên cùng split; chọn bằng validation.")
print("Chỉ số: macro-F1, precision/recall/F1/support đủ 8 lớp và confusion matrix 8×8.")
print("Dữ liệu thang gốc cho cây: X_*_raw.npy; nhánh log/scale: X_*_scaled.npy hoặc .pkl.")
print("Mapping mới dùng label_mapping.json; không dùng label_encoder/scaler cũ của lần chạy trước.")
